# <Paper title>: reproduction and extension

**Student:** <name> · **Paper:** <authors, title, venue, year, link> · **Track:** Docker (ns-3 / Python) or NixNet

**Summary (3-5 sentences):** what the paper claims, what you reproduced, what you changed, and the main result.

> **How to use this template** (delete this box before submitting)
> 1. Keep the sections of the track you use; delete the example sections you do not need.
> 2. Replace the example scenario (`ns3/aanet`, `sim/main.py` or the experiment in `flake.nix`) with your own.
> 3. Every experiment runs through `runner.py`, which caches results in `results/`. Re-running a cell only runs what is missing.
> 4. Before submitting: commit, clone into a fresh folder, follow the README and run this notebook top to bottom (*Kernel → Restart & Run All*).

## 0. Setup
`ENGINE="docker"` runs every job in the pinned Docker images and is required for the results you submit.
`ENGINE="local"` runs the same jobs without Docker and is only for debugging.

In [ ]:
import os

os.environ.setdefault("ENGINE", "docker")  # set before importing runner
TRACKS = os.environ.get("TRACKS", "ns3,python,nixnet").split(",")  # delete what you do not use

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import runner
from runner import grid, read_iperf3, read_ping, run_nixnet, run_ns3, run_py, sweep

FIG = Path("figures")
FIG.mkdir(exist_ok=True)


def mean_ci(df, by, col):
    """Mean of `col` over seeds with an approximate 95% confidence interval (1.96 x SEM)."""
    g = df.groupby(by)[col]
    out = g.agg(["mean", "std", "count"])
    out["ci95"] = 1.96 * out["std"] / np.sqrt(out["count"])
    return out.reset_index()


pd.Series(runner.environment_info(), name="value").to_frame()  # report this with your results

## 1. Experiment design
- **Research question:** <what the experiment answers>
- **Parameters swept:** <names, values, why these values>
- **Fixed parameters:** <and where they come from in the paper>
- **Metrics:** <definitions and units>
- **Repetitions:** <number of seeds/runs and why it is enough>

## 2. ns-3 experiment (Docker track)
Example: AANET snapshot (`ns3/aanet/aanet.cc`). Aircraft in a 1000 km x 1000 km area, air-to-air links within `commRange`,
min-hop routing, 10 random UDP flows. Each job is one `(params, seed)` pair; the seed is passed to ns-3 as `--RngRun`.

In [ ]:
if "ns3" in TRACKS:
    ns3_configs = grid(numNodes=[50, 100], commRange=[200_000, 300_000, 400_000], simTime=[10])
    ns3_seeds = range(1, 6)
    ns3_df = sweep(run_ns3, ns3_configs, ns3_seeds)  # one row per flow, per job
    display(ns3_df.head())

In [ ]:
if "ns3" in TRACKS:
    # aggregate flows per run first, then compute mean and CI over seeds
    per_run = (ns3_df.groupby(["numNodes", "commRange", "seed"])
               .agg(reachable=("reachable", "mean"), delay_ms=("mean_delay_ms", "mean"))
               .reset_index())
    fig, axes = plt.subplots(1, 2, figsize=(10, 3.8))
    for ax, col, label in [(axes[0], "reachable", "Reachable flows (fraction)"),
                           (axes[1], "delay_ms", "Mean end-to-end delay [ms]")]:
        s = mean_ci(per_run, ["numNodes", "commRange"], col)
        for n, g in s.groupby("numNodes"):
            ax.errorbar(g.commRange / 1e3, g["mean"], yerr=g.ci95, marker="o", capsize=3, label=f"{n} aircraft")
        ax.set_xlabel("Air-to-air range [km]")
        ax.set_ylabel(label)
        ax.grid(alpha=0.3)
    axes[0].legend()
    fig.tight_layout()
    fig.savefig(FIG / "ns3_reachability_delay.pdf", bbox_inches="tight")

## 3. Python simulation (Docker track)
Example: Monte Carlo of fleet connectivity to two ground stations (`sim/main.py`). The script receives its parameters as JSON
and the seed on the command line, so every run is reproducible.

In [ ]:
if "python" in TRACKS:
    py_configs = grid(n_aircraft=[50, 100, 200], range_km=[200, 300, 400])
    py_seeds = range(10)
    py_df = sweep(run_py, py_configs, py_seeds)  # one row per job
    display(py_df.head())

In [ ]:
if "python" in TRACKS:
    s = mean_ci(py_df, ["n_aircraft", "range_km"], "frac_connected")
    fig, ax = plt.subplots(figsize=(5, 3.8))
    for n, g in s.groupby("n_aircraft"):
        ax.errorbar(g.range_km, g["mean"], yerr=g.ci95, marker="o", capsize=3, label=f"{n} aircraft")
    ax.set_xlabel("Air-to-air range [km]")
    ax.set_ylabel("Aircraft with a path to ground")
    ax.grid(alpha=0.3)
    ax.legend()
    fig.savefig(FIG / "py_connectivity.pdf", bbox_inches="tight")

## 4. NixNet experiment (NixNet track)
Example: ground station → aircraft relay → ground station (`flake.nix`). Each air link adds a one-way delay `d` (and optional loss),
so the expected RTT is about 4·d. Experiments are named `relay-d<delay>-l<loss>`; each run is stored in `results/nixnet/<name>/<run>/`.
Start Jupyter with `nix develop -c jupyter lab` so that `nix` is available.

In [ ]:
if "nixnet" in TRACKS:
    rows = []
    for exp in [f"relay-d{d}-l{l}" for d in (1, 5, 20) for l in (0, 1)]:
        for run_dir in run_nixnet(exp, runs="1-3"):
            client = run_dir / "gs-west"
            rows.append({"experiment": exp, "run": run_dir.name,
                         **read_ping(client / "ping.txt"), **read_iperf3(client / "iperf3.json")})
    nix_df = pd.DataFrame(rows)
    nix_df[["delay_ms", "loss_pct_cfg"]] = nix_df.experiment.str.extract(r"d(\d+)-l(\d+)").astype(int)
    display(nix_df.head())

In [ ]:
if "nixnet" in TRACKS:
    fig, axes = plt.subplots(1, 2, figsize=(10, 3.8))
    for loss, g in nix_df.groupby("loss_pct_cfg"):
        s = mean_ci(g, "delay_ms", "rtt_avg_ms")
        axes[0].errorbar(s.delay_ms, s["mean"], yerr=s.ci95, marker="o", capsize=3, label=f"loss {loss}%")
        s = mean_ci(g, "delay_ms", "throughput_mbps")
        axes[1].errorbar(s.delay_ms, s["mean"], yerr=s.ci95, marker="o", capsize=3, label=f"loss {loss}%")
    d = np.array(sorted(nix_df.delay_ms.unique()))
    axes[0].plot(d, 4 * d, "k--", lw=1, label="expected 4·d")
    axes[0].set_ylabel("RTT [ms]")
    axes[1].set_ylabel("TCP throughput [Mbit/s]")
    for ax in axes:
        ax.set_xlabel("One-way delay per air link [ms]")
        ax.grid(alpha=0.3)
        ax.legend()
    fig.tight_layout()
    fig.savefig(FIG / "nixnet_rtt_throughput.pdf", bbox_inches="tight")

## 5. Results and discussion
- **Reproduction:** which results of the paper you reproduced, and how closely (numbers, figures side by side).
- **Differences:** where your results differ and why (model simplifications, parameters, randomness).
- **Extension:** what you changed or added, and what you learned from it.
- **Limitations:** what your experiment cannot show.

## 6. Reproducibility
- **Environment:** the table printed in section 0 (engine, image IDs or Nix version, git commit).
- **How to reproduce:** see `README.md`. Results are cached in `results/` with a `.json` file per job describing how it was produced.
- **Checklist before submitting:**
  - [ ] all results produced with `ENGINE="docker"` (Docker track) or via `nix develop` (NixNet track)
  - [ ] `flake.lock` / pinned `requirements*.txt` committed
  - [ ] fresh clone + README + *Restart & Run All* works
  - [ ] git working tree clean (`git_dirty` is `False` above)